# YOLO12s + Direct EMA + GhostConv — frozen proposed-model training

Notebook Kaggle ini melatih arsitektur proposed yang sudah dibekukan secara eksperimen. Arsitektur, semantic pretrained transfer, dan regression test berasal dari tag Git immutable research/yolo12s-ema-ghost-architecture-v1 pada commit d199dd3bac9dd70875370ed3ba16a8a22da1a21b.

Aktifkan GPU dan Internet di Kaggle. Runner meng-clone tag tersebut secara eksplisit, memakai trainer Ultralytics normal, dan memverifikasi semantic transfer untuk target RDD2022 lima kelas sebelum epoch pertama dimulai. Tidak ada remap manual, custom trainer, atau gradient checker yang dapat mengubah fairness eksperimen.


In [ ]:
# 1. Clone snapshot arsitektur yang dibekukan dan install repository lokal.
import json
import platform
import subprocess
import sys
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
FROZEN_TAG = 'research/yolo12s-ema-ghost-architecture-v1'
FROZEN_COMMIT = 'd199dd3bac9dd70875370ed3ba16a8a22da1a21b'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022-ema-ghost'

def log_section(title: str) -> None:
    print()
    print('=' * 88)
    print(title)
    print('=' * 88)

def run(command, cwd=None) -> None:
    print('>>', subprocess.list2cmdline([str(item) for item in command]))
    subprocess.run(command, cwd=cwd, check=True)

log_section('CLONE FROZEN PROPOSED ARCHITECTURE')
if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', FROZEN_TAG, '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', '--tags', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', '--detach', FROZEN_TAG], cwd=REPO_DIR)

REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert REPO_COMMIT == FROZEN_COMMIT, f'Unexpected source revision: {REPO_COMMIT}'
run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(REPO_DIR)])
sys.path.insert(0, str(REPO_DIR))

REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(
    f'repository={REPO_URL}\ntag={FROZEN_TAG}\ncommit={REPO_COMMIT}\n', encoding='utf-8'
)

import torch
import ultralytics

assert torch.cuda.is_available(), 'Aktifkan GPU accelerator di Kaggle sebelum training.'
DEVICE = 0
log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Frozen tag  : {FROZEN_TAG}')
print(f'Commit      : {REPO_COMMIT}')
print(f'GPU         : {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Dataset dan hyperparameter. Pertahankan protokol yang sama dengan baseline.
import zipfile

from ultralytics.utils import YAML

DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd-2022/datasets-china-split-fix')
DATA_YAML = WORKDIR / 'ch_rdd2022_frozen.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12s-ema-ghost.yaml'
RUNS_DIR = WORKDIR / 'runs'
EXPERIMENT_NAME = 'yolo12s_ema_ghost_semantic_pretrained_ch_rdd2022'

EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED = 0, 2, 42

assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
assert MODEL_YAML.exists(), f'Model YAML tidak ditemukan: {MODEL_YAML}'
for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / split / 'images'
    assert image_dir.exists(), f'Image directory tidak ditemukan: {image_dir}'
    image_count = sum(
        path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'} for path in image_dir.rglob('*')
    )
    assert image_count > 0, f'Tidak ada image pada split {split}'
    print(f'{split:5s}: {image_count:,} images')

DATA_YAML.write_text(
    f'''path: {DATA_ROOT.as_posix()}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''',
    encoding='utf-8',
)

semantic_spec = YAML.load(MODEL_YAML)['semantic_pretrained']
assert semantic_spec['source_layers'] == 22
assert semantic_spec['source_detect_from'] == [14, 17, 20]
assert semantic_spec['target_detect_from'] == [15, 19, 22]

EXPECTED_TRANSFER = {
    'mode': 'semantic',
    'source_tensors': 699,
    'target_tensors': 715,
    'semantic_candidates': 679,
    'exact_transferred_tensors': 673,
    'destination_missing_tensors': 8,
    'shape_mismatch_tensors': 6,
    'new_module_tensors': 36,
    'uninitialized_target_tensors': 42,
    'exact_transferred_parameter_tensors': 340,
    'exact_transferred_parameter_elements': 8515088,
}
RUNS_DIR.mkdir(parents=True, exist_ok=True)
RUN_MANIFEST = WORKDIR / f'{EXPERIMENT_NAME}_planned_run.json'
RUN_MANIFEST.write_text(
    json.dumps(
        {
            'repository': REPO_URL,
            'frozen_tag': FROZEN_TAG,
            'frozen_commit': REPO_COMMIT,
            'model_yaml': str(MODEL_YAML),
            'dataset_yaml': str(DATA_YAML),
            'epochs': EPOCHS,
            'imgsz': IMGSZ,
            'batch': BATCH,
            'nbs': NBS,
            'optimizer': OPTIMIZER,
            'lr0': LR0,
            'momentum': MOMENTUM,
            'weight_decay': WEIGHT_DECAY,
            'patience': PATIENCE,
            'seed': SEED,
            'expected_pretrained_transfer': EXPECTED_TRANSFER,
        },
        indent=2,
    ),
    encoding='utf-8',
)
print(f'Dataset YAML: {DATA_YAML}')
print(json.dumps(EXPECTED_TRANSFER, indent=2))


In [ ]:
# 3. Validasi graph proposed lima kelas sebelum trainer normal dijalankan.
from ultralytics import YOLO
from ultralytics.nn.modules import EMA, GhostConv
from ultralytics.nn.tasks import DetectionModel

log_section('PROPOSED MODEL PRE-FLIGHT')
inspection_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=False)
ema_layers = [layer.i for layer in inspection_model.model if isinstance(layer, EMA)]
ghost_layers = [layer.i for layer in inspection_model.model if isinstance(layer, GhostConv)]
assert ema_layers == [15, 19], f'Unexpected EMA layers: {ema_layers}'
assert ghost_layers == [16, 20], f'Unexpected GhostConv layers: {ghost_layers}'
assert inspection_model.model[-1].nc == 5
assert inspection_model.model[-1].f == [15, 19, 22]
print(f'EMA layers       : {ema_layers}')
print(f'GhostConv layers : {ghost_layers}')
print(f'Detect inputs    : {inspection_model.model[-1].f}')
inspection_model.info(detailed=False, verbose=True, imgsz=IMGSZ)
del inspection_model
torch.cuda.empty_cache()

# Jangan memanggil model.load() di sini. Trainer di cell berikut membangun target nc=5
# dan memanggil BaseModel.load() sekali dengan pretrained='yolo12s.pt'.
model = YOLO(str(MODEL_YAML), task='detect')


In [ ]:
# 4. Training normal dengan checkpoint resmi dan audit transfer semantic pada awal trainer.
TRAINING_TRANSFER_REPORT = {}
TRANSFER_REPORT_PATH = None

def verify_semantic_pretrained_transfer(trainer) -> None:
    global TRANSFER_REPORT_PATH
    report = dict(getattr(trainer.model, 'pretrained_transfer_report', {}))
    differences = {
        key: {'expected': expected, 'actual': report.get(key)}
        for key, expected in EXPECTED_TRANSFER.items()
        if report.get(key) != expected
    }
    if differences:
        raise AssertionError(f'Semantic pretrained audit failed: {json.dumps(differences, indent=2)}')
    TRAINING_TRANSFER_REPORT.clear()
    TRAINING_TRANSFER_REPORT.update(report)
    TRANSFER_REPORT_PATH = Path(trainer.save_dir) / 'semantic_pretrained_transfer.json'
    TRANSFER_REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
    log_section('SEMANTIC PRETRAINED TRANSFER VERIFIED')
    print(json.dumps(report, indent=2))

model.add_callback('on_pretrain_routine_end', verify_semantic_pretrained_transfer)
log_section('TRAINING STARTED — FROZEN YOLO12S + DIRECT EMA + GHOSTCONV')
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}')
# exist_ok=False mencegah artefak eksperimen lama tercampur; ubah EXPERIMENT_NAME untuk trial baru.
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    nbs=NBS,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=EXPERIMENT_NAME,
    exist_ok=False,
    pretrained='yolo12s.pt',
    optimizer=OPTIMIZER,
    lr0=LR0,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    cos_lr=False,
    patience=PATIENCE,
    seed=SEED,
    plots=True,
    verbose=True,
)
assert TRAINING_TRANSFER_REPORT, 'Transfer report tidak pernah diterima oleh callback trainer.'
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = Path(model.trainer.best)
LAST_PT = Path(model.trainer.last)
assert BEST_PT.exists() and LAST_PT.exists(), 'Checkpoint training tidak lengkap.'
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')


In [ ]:
# 5. Evaluasi best.pt, simpan metadata, lalu buat ZIP artefak Kaggle.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

val_metrics = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=f'{EXPERIMENT_NAME}_val',
    exist_ok=True,
    plots=True,
)
VAL_DIR = Path(val_metrics.save_dir)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics)}

test_label_dir = DATA_ROOT / 'test' / 'labels'
if test_label_dir.exists() and any(test_label_dir.rglob('*.txt')):
    test_metrics = best_model.val(
        data=str(DATA_YAML),
        split='test',
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_test',
        exist_ok=True,
        plots=True,
    )
    TEST_DIR = Path(test_metrics.save_dir)
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
else:
    TEST_DIR = RUNS_DIR / f'{EXPERIMENT_NAME}_test_predictions'
    best_model.predict(
        source=str(DATA_ROOT / 'test' / 'images'),
        imgsz=IMGSZ,
        device=DEVICE,
        conf=0.25,
        save=True,
        save_txt=True,
        project=str(RUNS_DIR),
        name=TEST_DIR.name,
        exist_ok=True,
    )
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_DIR)}

EVALUATION_REPORT['pretrained_transfer'] = TRAINING_TRANSFER_REPORT
EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
FINAL_MANIFEST = WORKDIR / f'{EXPERIMENT_NAME}_run_manifest.json'
FINAL_MANIFEST.write_text(
    json.dumps(
        {
            'repository': REPO_URL,
            'frozen_tag': FROZEN_TAG,
            'frozen_commit': REPO_COMMIT,
            'model_yaml': str(MODEL_YAML),
            'dataset_yaml': str(DATA_YAML),
            'hyperparameters': {
                'epochs': EPOCHS,
                'imgsz': IMGSZ,
                'batch': BATCH,
                'nbs': NBS,
                'optimizer': OPTIMIZER,
                'lr0': LR0,
                'momentum': MOMENTUM,
                'weight_decay': WEIGHT_DECAY,
                'patience': PATIENCE,
                'seed': SEED,
            },
            'pretrained_transfer': TRAINING_TRANSFER_REPORT,
            'best_checkpoint': str(BEST_PT),
            'last_checkpoint': str(LAST_PT),
            'evaluation': EVALUATION_REPORT,
        },
        indent=2,
    ),
    encoding='utf-8',
)

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_path(archive: zipfile.ZipFile, path: Path, added: set) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    count = 0
    for file_path in files:
        archive_name = file_path.relative_to(WORKDIR).as_posix()
        if archive_name not in added:
            archive.write(file_path, archive_name)
            added.add(archive_name)
            count += 1
    return count

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    added = set()
    file_count = sum(
        add_path(archive, path, added)
        for path in (
            RUN_DIR,
            VAL_DIR,
            TEST_DIR,
            DATA_YAML,
            MODEL_YAML,
            REPO_METADATA,
            RUN_MANIFEST,
            FINAL_MANIFEST,
            TRANSFER_REPORT_PATH,
            EVALUATION_JSON,
        )
    )

print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created: {ZIP_PATH} ({file_count} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
